In [ ]:
# Extract and compile datasets

import pandas as pd

def extract(date,i,n):
    """Extract the dataset from an individual file as a pandas DataFrame."""
    # Construct the file suffix, accounting for single-digit and double-digit part numbers
    if n > 9:
        z = ""
    else:
        z = "0"
    # Read DataFrame
    ufdf = pd.read_stata(f'output_{str(date[i])}_part-000{z+str(n)}.dta')
    return ufdf

# Dates corresponding to the selected dataset snapshots
date = (
    '2022_10_2', '2023_1_1', '2023_4_2', '2023_7_5',
    '2023_10_1', '2024_1_7', '2024_4_7', '2024_7_7',
    '2024_10_6', '2025_1_5', '2025_4_6', '2025_7_6'
)

for i in range(0,len(date)):
    ufdf = []
    # Define the range of dataset parts for each snapshot
    if i == 0:
        x=0
        y=7
    elif i == 1:
        x=2
        y=10
    elif i in (6,7,8,10):
        x=0
        y=11
    elif i == 11:
        x=0
        y=12
    else:
        x=0
        y=10

    for n in range(x,y):
        ufdf.append(extract(date,i,n))

    # Combine dataset parts into a single DataFrame
    ufdf = pd.concat(ufdf, ignore_index=True)

    print(f"Unfiltered dataset for {date[i]} of length: {len(ufdf)}")

    # Save the dataset for the current snapshot
    ufdf.to_csv(f"ufdf_{date[i]}.csv", index=False)

print("\nExtraction completed.")

KeyboardInterrupt: 

In [5]:
# Extract a 12,000-sample dataset (combined)

import pandas as pd

# Dates corresponding to the selected dataset snapshots
date = (
    '2022_10_2', '2023_1_1', '2023_4_2', '2023_7_5',
    '2023_10_1', '2024_1_7', '2024_4_7', '2024_7_7',
    '2024_10_6', '2025_1_5', '2025_4_6', '2025_7_6'
)

sample_df=[]

for i in range(0,len(date)):
    df = pd.read_csv(f"ufdf_{date[i]}.csv")
    # Randomly sample 1000 records from each snapshot date
    sample = df.sample(n=1000, random_state=42)
    # Record the snapshot date/quarter associated with each record
    sample["quarter"] = date[i]
    sample_df.append(sample)

    print(f"Sampling {date[i]}")

# Combine samples from all snapshot dates into a single dataset
df_all = pd.concat(sample_df, ignore_index=True)
# Save the combined 12000-sample
df_all.to_csv("ufdf_all.csv", index=False)

print("\nSample created!")

Sampling 2022_10_2
Sampling 2023_1_1
Sampling 2023_4_2
Sampling 2023_7_5
Sampling 2023_10_1
Sampling 2024_1_7
Sampling 2024_4_7
Sampling 2024_7_7
Sampling 2024_10_6
Sampling 2025_1_5
Sampling 2025_4_6
Sampling 2025_7_6

Sample created!


In [6]:
# VSRS

import pandas as pd
import re

def vsrs(description):
    """Extract visa/sponsorship/right-to-work statements (VSRS)."""

    keywords = [
        "visa",
        "sponsor",
        "right to live and work",
        "right to work",
        "eligible to work",
        "eligibility to work",
        "work eligibility",
        "authorised to work",
        "authorisation to work",
        "authorized to work",
        "authorization to work",
        "working right",
        "work authorisation",
        "work authorization",
        "permission to work",
        "work permit",
        "working permit",
        "work permission",
        "working permission",
    ]

    statements = []

    # Split the description into sentences and bullet-point statements
    sentences = re.split(r'(?<=[.!?])\s+|\s*[-*+·•]\s*', description)

    # Find and store statements containing at least one VSRS keyword
    for sentence in sentences:
        if any(keyword in sentence.lower() for keyword in keywords):
            statements.append(sentence.strip())

    # Return combined statement(s) and remove duplicates (preserving original order)
    return " ".join(dict.fromkeys(statements)) or "Not found"

# Extract VSRS statements to new column and create a separate dataset containing only jobs with VSRS
df = pd.read_csv("ufdf_all.csv")
df["vsrs"] = df["description"].apply(vsrs)
vsrs_df = df[df["vsrs"] != "Not found"].copy()

# Determining the number of jobs with VSRS
vsrs_jobs = (df["vsrs"] != "Not found").sum()
print(f"Jobs with VSRS: {vsrs_jobs} of {len(df)}")

# Save the complete dataset with the VSRS column
df.to_csv("ufdf_all_vsrs_applied.csv", index=False)
# Save only the jobs containing VSRS statements
vsrs_df.to_csv("ufdf_all_vsrs_filtered.csv", index=False)

Jobs with VSRS: 1141 of 12000
